# Part 3 Noise Label Identification 对脏crop数据的分label及neighbors识别清洗

In [ ]:
import httpx
#from bs4 import BeautifulSoup
import re
import json
import pandas as pd
import os
import random
import time
import openai
import requests
import torch
import sqlite3
from PIL import Image, ImageOps
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection,pipeline
from accelerate import Accelerator
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
load_dotenv(override=True)
from huggingface_hub import login
login(token=os.getenv("HUGGINGFACEHUB_API_TOKEN"))
current_dir = os.getcwd()
PROJECT_ROOT = os.path.dirname(os.path.dirname(current_dir))
db_path = os.path.join(PROJECT_ROOT, "data", "commons_image_manifest.sqlite")

def load_img_with_orientation(path):
    img = Image.open(path).convert("RGB")
    img = ImageOps.exif_transpose(img)  # 确保方向正确
    return img

OPENAI_MODEL_NAME = "gpt-5.4-mini"
LOCAL_VLM_NAME = "qwen/qwen3.6-27b"
LM_STUDIO_ENDPOINT = "http://127.0.0.1:1234"
PROJECT_ROOT


In [ ]:
with sqlite3.connect(db_path) as conn:
    count = pd.read_sql_query("SELECT COUNT(*) FROM crops GROUP BY series",conn)
count